# tac — Task-Aware Codec Training
**comma video compression challenge** | tac v0.8.0

Self-contained: mounts Google Drive for data persistence across sessions.
First run uploads code+data to Drive. Subsequent runs resume instantly.

**GPU required** — Change Runtime > T4 GPU

In [ ]:
# 1. Mount Google Drive for persistence
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/tac_training'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print(f'Working directory: {WORK}')
!ls

In [ ]:
# 2. Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU - change runtime!'

In [ ]:
# 3. First-time setup (skip if already done — checks for 'setup_done' marker)
import os
if not os.path.exists('setup_done'):
    print('=== First-time setup ===')
    # Install deps
    !pip install -q torch torchvision av safetensors timm einops segmentation-models-pytorch numpy pydantic

    # Clone upstream (scorer models + GT video)
    if not os.path.exists('upstream'):
        !git clone --depth 1 https://github.com/commaai/comma_video_compression_challenge.git upstream
        !cd upstream && git lfs pull

    # Upload tac package if not present
    if not os.path.exists('src/tac/__init__.py'):
        print('\n>>> Upload tac_complete.zip (958KB) from experiments/cloud_package/')
        print('>>> Generate it locally with: cd experiments/cloud_package && zip -r /tmp/tac_complete.zip src/ train_tac.py archive.zip')
        from google.colab import files
        uploaded = files.upload()
        !unzip -qo tac_complete.zip

    # Generate uniform saliency
    if not os.path.exists('saliency.npy'):
        import numpy as np
        np.save('saliency.npy', np.ones((30, 874, 1164), dtype=np.float32))

    !touch setup_done
    print('Setup complete!')
else:
    print('Setup already done. Installing deps for this session...')
    !pip install -q torch torchvision av safetensors timm einops segmentation-models-pytorch numpy pydantic

In [ ]:
# 4. Verify tac
import sys
sys.path.insert(0, 'src')
sys.path.insert(0, 'upstream')

import tac
from tac.architectures import build_postfilter
from tac.training import TrainConfig
from tac.scorer import detect_device

print(f'tac v{tac.__version__}, device: {detect_device()}')
model = build_postfilter('standard', hidden=64)
print(f'Model: {sum(p.numel() for p in model.parameters())} params')

# Check for resume state
import glob
states = glob.glob('weights/training_state_*.pt')
if states:
    print(f'Resume state found: {states}')
else:
    print('Fresh start (no resume state)')

In [ ]:
# 5. TRAIN — Standard h=64 (proven recipe)
# Resume-capable: if Colab disconnects, re-run this cell
# Training state saves every 50 epochs to Google Drive

import os
RESUME = ''
state_files = sorted(glob.glob('weights/training_state_*.pt'))
if state_files:
    RESUME = f'--resume-from {state_files[-1]}'
    print(f'Resuming from {state_files[-1]}')

!PYTHONPATH=src:upstream PYTHONUNBUFFERED=1 python train_tac.py \
    --tag colab_h64_standard \
    --hidden 64 \
    --epochs 2500 \
    --alpha 20 \
    --sal-lambda 1.0 \
    --subsample 4 \
    --output-dir ./weights \
    --archive ./archive.zip \
    --gt-video ./upstream/videos/0.mkv \
    --saliency ./saliency.npy \
    --models-dir ./upstream/models \
    --upstream-dir ./upstream \
    $RESUME

In [ ]:
# 5b. ALTERNATIVE: KL Distill (SegNet attack)
# Uncomment and run instead of cell 5

# !PYTHONPATH=src:upstream PYTHONUNBUFFERED=1 python train_tac.py \
#     --tag colab_h64_kl_distill \
#     --hidden 64 \
#     --epochs 2500 \
#     --loss-mode kl_distill \
#     --temperature-start 5.0 \
#     --temperature-end 1.0 \
#     --alpha 20 \
#     --sal-lambda 1.0 \
#     --subsample 4 \
#     --output-dir ./weights \
#     --archive ./archive.zip \
#     --gt-video ./upstream/videos/0.mkv \
#     --saliency ./saliency.npy \
#     --models-dir ./upstream/models \
#     --upstream-dir ./upstream

In [ ]:
# 6. Check results
import json, glob
for meta in sorted(glob.glob('weights/*_best_meta.json')):
    data = json.loads(open(meta).read())
    print(f"  {meta}: epoch {data['epoch']}, scorer {data['scorer']:.4f}, int8 {data['int8_size']} bytes")

In [ ]:
# 7. Download best checkpoint (also saved to Google Drive automatically)
from google.colab import files
for f in glob.glob('weights/*best_int8*'):
    print(f'Downloading {f}')
    files.download(f)
for f in glob.glob('weights/*best_meta*'):
    files.download(f)